In [ ]:
# Less Salt Less Sugar Monthly Report (launch from 31Aug2026. last for 1 year at least)
#     1. Daily Badge impression by device 
#     2. Daily Brand Page (LMS) Pageview/ Impressions   
#     3. Daily Theme Listing Impressions (adv search)

#http://192.168.61.59:8888/notebooks/reports/monthly_icon_impression_report/monthly_icon_impression_report_result.ipynb

In [2]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../../../BQ2.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [4]:
current_date = datetime.date.today()
first_day_of_current_month = datetime.date(current_date.year, current_date.month, 1)
last_day_of_previous_month = first_day_of_current_month - datetime.timedelta(days=1)

month = last_day_of_previous_month.month
year = last_day_of_previous_month.year

str_month = str(month)
if len(str_month)==1:
    str_month = '0'+str_month

all_dates = pd.date_range(
    start=f"{year}-{str_month}-01",
    end=pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0),
    freq="D",
)

str_month

'09'

In [5]:
template_file = 'LSLS_report_template.xlsx'
excel_name = template_file.replace('template.xlsx', '%s.xlsx' % datetime.date.today())
copyfile(template_file, excel_name)

# excel_name = "LSLS_report_2026-09-25.xlsx"

'LSLS_report_2026-10-02.xlsx'

In [ ]:
# 1. Daily Badge impression by device     DONE
#  --> monthly_icon_impression_report_result.ipynb cell 8 少鹽少糖食店 Icon Impression v2
# 
# #少鹽少糖食店 Icon Impression v2

sql = f'''

SELECT date(time) as querydate,platform,count(1) as count_ 
FROM `openrice-production.ORGA.PV_{year}{str_month}*` 
WHERE EventAction = 'impression.poi'
and cast(REGEXP_EXTRACT(lower(EventLabelRaw), r'poiid:(\d+)') as INT64) in (Select poiid FROM `openrice-production.openrice3.promotionpoi` WHERE PromotionId =11)
group by platform,querydate
    '''

df_big_query = client.query(sql).result().to_dataframe()


<>:13: SyntaxWarning: invalid escape sequence '\d'
<>:13: SyntaxWarning: invalid escape sequence '\d'
C:\Users\lenalee\AppData\Local\Temp\ipykernel_81040\1521017806.py:13: SyntaxWarning: invalid escape sequence '\d'
  '''
C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [8]:
# df_big_query

In [9]:
pivoted_df = df_big_query.pivot(index='querydate', columns='platform', values='count_')
pivoted_df = pivoted_df.fillna(0)
pivoted_df["Web"]=pivoted_df["desktop"]
pivoted_df["Mobile Web"]=pivoted_df["mobile"]
try:
    pivoted_df["Android"]=pivoted_df["android"]+pivoted_df["hms"]
except:
    pivoted_df["Android"]=pivoted_df["android"]
    
pivoted_df["IOS"]=pivoted_df["ios"]
pivoted_df =pivoted_df[["Web","Mobile Web","Android","IOS"]]

pivoted_df.index.name = None
pivoted_df.reset_index(inplace=True)

pivoted_df

platform,index,Web,Mobile Web,Android,IOS
0,2026-09-01,41168,26673,63722,310784
1,2026-09-02,44358,29954,68532,329706
2,2026-09-03,45891,31416,70424,344449
3,2026-09-04,50407,32254,76952,386028
4,2026-09-05,20598,38190,96330,460941
5,2026-09-06,18177,37100,89108,445521
6,2026-09-07,47356,28485,66367,325432
7,2026-09-08,49089,29809,68809,346070
8,2026-09-09,47933,29237,67531,343454
9,2026-09-10,46813,28872,70961,341979


In [10]:
with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    pivoted_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=2, header=None, index=False)

In [46]:
# 2. Daily Brand Page (LMS) Pageview
# web: view.SR1.Promotion | CityID:0;PromotionId:13,13,13,13,13,13,13,13,13,13,13,13,13,13,13;Sn:https://www.openrice.com/zh/hongkong/l-%E5%B0%91%E9%B9%BD%E5%B0%91%E7%B3%96%E9%A3%9F%E5%BA%97-l35336
# web: view.SR1.Promotion | CityID:0;PromotionId:13,13,13,13,13,13,13,13,13,13,13,13,13,13,13;Sn:https://www.openrice.com/zh/hongkong/l-%E5%B0%91%E9%B9%BD%E5%B0%91%E7%B3%96%E9%A3%9F%E5%BA%97-l35336?sortBy=ORScoreDesc
# web: or.search.adv | CityID:0;Sr:Biz;SortBy:ORScoreDesc;LandmarkId:35336;UiCity:hongkong;Sn:https://www.openrice.com/zh/hongkong/l-%E5%B0%91%E9%B9%BD%E5%B0%91%E7%B3%96%E9%A3%9F%E5%BA%97-l35336
# app: 11:25:26|| or.search.layer.search| CityID:0;geo:22.2915762%2C114.2081518;LndID:35336;Page:1;sr:lmsSr1;Lang:zh_TW;Ver:7.20.4; sn:hk.Search.layer
# app: 16:31:34|| or.search.quick| CityID:0;LndID:35336;Page:1;Lang:zh_TW;Ver:7.21.0; sn:hk.QckSearch
# (nn) app: 11:25:26|| view.SR1.Promotion| CityID:0;PromotionID:13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C12%2C13%2C13%2C13%2C13%2C13%2C13;;Lang:zh_TW;Ver:7.20.4; sn:hk.LMS2.35336.tab.-990.1

sql = f"""
    select date(time) as querydate, platform, count(1) as count
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE (
        ( (LOWER(EventAction) = 'view.sr1.promotion') and (LOWER(EventLabelRaw) LIKE '%35336%') and (LOWER(platform) in ('desktop','mobile'))  )   -- web
        OR
        ( (LOWER(EventAction) = 'or.search.adv') and (LOWER(EventLabelRaw) LIKE '%35336%') and (LOWER(platform) in ('desktop','mobile'))   )       -- web
        OR 
        ( (LOWER(EventAction) = 'or.search.layer.search')   and    (LOWER(EventLabelRaw) LIKE '%35336%' )   )         -- app
        OR
        ( (LOWER(EventAction) = 'or.search.quick')   and    (LOWER(EventLabelRaw) LIKE '%35336%' )   )         -- app   

    )
    group by platform, querydate
    """

df_big_query_2_2 = client.query(sql).result().to_dataframe()


C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [38]:
# df_big_query_2_2

In [47]:
pivoted_df_2 = df_big_query_2_2.pivot(index='querydate', columns='platform', values='count')
pivoted_df_2 = pivoted_df_2.reindex(index=all_dates, fill_value=0).fillna(0)

expected_platforms = ["desktop", "mobile", "ios", "android", "hms"]
pivoted_df_2 = pivoted_df_2.reindex(columns=expected_platforms, fill_value=0)

pivoted_df_2["Web"]=pivoted_df_2["desktop"] + pivoted_df_2["mobile"]
pivoted_df_2["App"]=pivoted_df_2["ios"] + pivoted_df_2["android"] + pivoted_df_2["hms"]
pivoted_df_2 =pivoted_df_2[["Web","App"]]

pivoted_df_2.index.name = None
pivoted_df_2.reset_index(inplace=True)
pivoted_df_2["index"] = pivoted_df_2["index"].dt.date
pivoted_df_2

platform,index,Web,App
0,2026-09-01,174,129
1,2026-09-02,141,198
2,2026-09-03,125,193
3,2026-09-04,107,152
4,2026-09-05,70,230
5,2026-09-06,59,227
6,2026-09-07,143,167
7,2026-09-08,104,213
8,2026-09-09,137,163
9,2026-09-10,121,174


In [13]:
with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    pivoted_df_2.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=37, header=None, index=False)

In [32]:
# 3. Daily Theme Listing Impressions (adv search)

# web: view.SR1.Promotion | CityID:0;PromotionId:13,13,13,13,13,13,13,13,13,13,13,13,13,13,13;Sn:https://www.openrice.com/zh/hongkong/restaurants/type/%E5%B0%91%E9%B9%BD%E5%B0%91%E7%B3%96%E9%A3%9F%E5%BA%97?sortBy=ORScoreDesc
# app: 11:26:28|| or.search.adv| CityID:0;geo:22.2915762%2C114.2081518;LndID:35336;Page:1;sr:lmsSr1;Lang:zh_TW;Ver:7.20.4; sn:hk.AdvSearch
# (nn) app: 11:26:28|| view.SR1.Promotion| CityID:0;PromotionID:13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C12%2C13%2C13%2C13%2C13%2C13%2C13;;Lang:zh_TW;Ver:7.20.4; sn:hk.LMS2.35336.tab.-990.1

sql = f"""
    select date(time) as querydate, platform, count(1) as count
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE (
        (       (LOWER(EventAction) = 'view.sr1.promotion') 
                    and  
                REGEXP_CONTAINS(
                    LOWER(EventLabelRaw),
                    r'/restaurants/type/%e5%b0%91%e9%b9%bd%e5%b0%91%e7%b3%96%e9%a3%9f%e5%ba%97'
                                ) 
        )  -- web
        OR
        (   (LOWER(EventAction) = 'or.search.adv') and (LOWER(EventLabelRaw) LIKE '%35336%') and (LOWER(platform) in ('android','ios','hms'))   )   -- app
    )
    group by platform, querydate
    """

df_big_query_3 = client.query(sql).result().to_dataframe()

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [40]:
# df_big_query_3

In [33]:
pivoted_df_3 = df_big_query_3.pivot(index='querydate', columns='platform', values='count')
pivoted_df_3 = pivoted_df_3.reindex(index=all_dates, fill_value=0).fillna(0)
expected_platforms = ["desktop", "mobile", "ios", "android", "hms"]
pivoted_df_3 = pivoted_df_3.reindex(columns=expected_platforms, fill_value=0)

pivoted_df_3["Web"]= pivoted_df_3["desktop"] + pivoted_df_3["mobile"]
pivoted_df_3["App"]=pivoted_df_3["ios"] + pivoted_df_3["android"] + pivoted_df_3["hms"]
pivoted_df_3 =pivoted_df_3[["Web","App"]]


pivoted_df_3.index.name = None
pivoted_df_3.reset_index(inplace=True)
pivoted_df_3["index"] = pivoted_df_3["index"].dt.date
pivoted_df_3

platform,index,Web,App
0,2026-09-01,0,30
1,2026-09-02,0,37
2,2026-09-03,0,32
3,2026-09-04,0,34
4,2026-09-05,0,35
5,2026-09-06,0,30
6,2026-09-07,0,55
7,2026-09-08,0,29
8,2026-09-09,0,32
9,2026-09-10,0,25


In [16]:
with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    pivoted_df_3.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=72, header=None, index=False)

Below is Testing!!!

In [8]:
sql=f"""

SELECT
  Time,
  EventCategory,
  Platform,
  Product,
  EventLabelRaw,
  EventData
FROM `openrice-production.ORGA.PV_20260924`
WHERE EventAction = 'view.SR1.Promotion'
  AND LOWER(COALESCE(EventLabelRaw, '')) LIKE '%l35336%'
  AND LOWER(COALESCE(EventLabelRaw, '')) LIKE '%cityid:0%'
  AND LOWER(COALESCE(EventLabelRaw, '')) LIKE '%promotionid:13%'
  AND LOWER(COALESCE(EventLabelRaw, '')) NOT LIKE '%tracking%'
ORDER BY Time DESC
LIMIT 100;

"""
df_big_query_4 = client.query(sql).result().to_dataframe()
df_big_query_4


C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,Time,EventCategory,Platform,Product,EventLabelRaw,EventData
0,2026-09-24 22:49:55.700000+00:00,WebEvent,desktop,OpenRice,"CityID:0;PromotionId:13,100,13,13,13,13,13,13,...",https://www.openrice.com/zh/hongkong/l-少鹽少糖食店-...
1,2026-09-24 22:45:08.100000+00:00,WebEvent,desktop,OpenRice,"CityID:0;PromotionId:13,13,13,13,13,13,13,13,1...",https://www.openrice.com/zh/hongkong/l-少鹽少糖食店-...
2,2026-09-24 22:22:09.300000+00:00,WebEvent,desktop,OpenRice,"CityID:0;PromotionId:13,13,13,13,13;Sn:https:/...",https://www.openrice.com/zh/hongkong/l-少鹽少糖食店-...
3,2026-09-24 22:21:48.200000+00:00,WebEvent,desktop,OpenRice,"CityID:0;PromotionId:13,13,13,13,13;Sn:https:/...",https://www.openrice.com/zh/hongkong/l-少鹽少糖食店-...
4,2026-09-24 22:00:48.800000+00:00,WebEvent,desktop,OpenRice,"CityID:0;PromotionId:13,13,13,13,13,13,13,13,1...",https://www.openrice.com/zh/hongkong/l-少鹽少糖食店-...
5,2026-09-24 21:07:20.100000+00:00,WebEvent,bot,OpenRice,"CityID:0;PromotionId:13,13,13,13,13,13,13,13,1...",https://www.openrice.com/zh/hongkong/l-少鹽少糖食店-...
6,2026-09-24 20:21:36.500000+00:00,WebEvent,bot,OpenRice,"CityID:0;PromotionId:13,13,13,13,13,13,13,13,1...",https://www.openrice.com/zh/hongkong/l-少鹽少糖食店-...
7,2026-09-24 18:11:13.200000+00:00,WebEvent,desktop,OpenRice,"CityID:0;PromotionId:13,13,13,13,13,13,13,13,1...",https://www.openrice.com/zh/hongkong/l-少鹽少糖食店-...
8,2026-09-24 17:46:15.400000+00:00,WebEvent,mobile,OpenRice,"CityID:0;PromotionId:13,13,13,13,13,13,13,13,1...",https://www.openrice.com/zh/hongkong/l-少鹽少糖食店-...
9,2026-09-24 16:37:04.500000+00:00,WebEvent,desktop,OpenRice,"CityID:0;PromotionId:13,13,13,13,13,13,13,13,1...",https://www.openrice.com/zh/hongkong/l-少鹽少糖食店-...


In [9]:
df_big_query_4.to_csv('df_big_query_4_1.csv', index=False)